Módulo: Limpieza y Depuración de Datos
Autor: Hector Pérez, Manuel González.
Descripción: Este módulo se encarga de la estandarización de formatos,
             tratamiento de valores nulos, eliminación de duplicados
             y validación de tipos de datos antes del procesamiento.


#Importación de Librerias

In [ ]:
# Procesamiento y análisis
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Tiempo y medición de rendimiento
import time

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.functions import col

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Configuración de warnings
import warnings
warnings.filterwarnings("ignore")

## Montaje del Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Verificación y listado de objetos del drive.

In [ ]:
import os

# Lista el contenido de la raíz de tu Google Drive
print('Contenido de /content/drive/MyDrive/:')
for item in os.listdir('/content/drive/MyDrive/'):
    print(item)

Contenido de /content/drive/MyDrive/:
DOC ADMISION
Varios
20V06
21V06
22V03
22V06
23v03
23V05
HECTOR ALEXIS PEREZ CIFUENTES
Colab Notebooks
24v05
CUN
EPICRISIS HECTOR PEREZ.pdf
26ET1
Fundamentos de inteligencia de negocios
MachineLearning
ACA1_Machine_Learning.docx
Herramientas de BIg Data
Google Meet
26ET3
25ET4


### Ruta al archivo data_residuos.xlsm

In [ ]:
ruta_residuos = "/content/drive/MyDrive/26ET3/PROYECTOII/data_residuos.xlsm"

## Crear sessión en Spark

In [ ]:
spark = SparkSession.builder \
    .appName("LimpiezaDatos") \
    .getOrCreate()

print("Sesión Spark iniciada correctamente")

Sesión Spark iniciada correctamente


In [ ]:
###Prueba para conocer la herramienta a utlizar para el manejo de los datos

# Lectura de la base nacional de residuos

# Guardamos el tiempo de inicio
inicio_pandas = time.time()

# Lectura de la base nacional de residuos
df_residuos_pd = pd.read_excel(ruta_residuos)

print("--- PANDAS ---")
print("Base cargada correctamente")
print(f"Cantidad de filas: {df_residuos_pd.shape[0]}")
print(f"Cantidad de columnas: {df_residuos_pd.shape[1]}")

# Guardamos el tiempo final y calculamos la diferencia
fin_pandas = time.time()
tiempo_total_pandas = fin_pandas - inicio_pandas
print(f"⏱️ Tiempo total de ejecución (Pandas): {tiempo_total_pandas:.4f} segundos\n")

--- PANDAS ---
Base cargada correctamente
Cantidad de filas: 136718
Cantidad de columnas: 13
⏱️ Tiempo total de ejecución (Pandas): 37.7284 segundos



In [ ]:
# Lectura de la base nacional de residuos por Spark, tipeo a string para poder cargar inicialmente
dfp_residuos_sp = pd.read_excel(ruta_residuos, dtype=str)

In [ ]:
# Guardamos el tiempo de inicio
inicio_spark = time.time()

df_residuos_sp = spark.createDataFrame(dfp_residuos_sp)

print("--- PYSPARK ---")
print("Base cargada correctamente")

# Conteo de filas y columnas en PySpark
print(f"Cantidad de filas: {df_residuos_sp.count()}")
print(f"Cantidad de columnas: {len(df_residuos_sp.columns)}")

# Guardamos el tiempo final y calculamos la diferencia
fin_spark = time.time()
tiempo_total_spark = fin_spark - inicio_spark
print(f"⏱️ Tiempo total de ejecución (PySpark): {tiempo_total_spark:.4f} segundos")

--- PYSPARK ---
Base cargada correctamente
Cantidad de filas: 136718
Cantidad de columnas: 13
⏱️ Tiempo total de ejecución (PySpark): 21.8852 segundos


## Convertimos a DoubleType las toneladas dispuestas

In [ ]:
df_residuos_sp = df_residuos_sp.withColumn(
    "Toneladas Dispuestas",
    col("Toneladas Dispuestas").cast(DoubleType())
)

print("Toneladas convertidas correctamente")

Toneladas convertidas correctamente


##Estandarisamos la fecha a dd-MM-yyyy

In [ ]:
from pyspark.sql.functions import to_date

df_residuos_sp = df_residuos_sp.withColumn(
    "Fecha de Cargue",
    to_date(
        col("Fecha de Cargue"),
        "dd-MM-yyyy"
    )
)

print("Fecha de Cargue convertida correctamente")

Fecha de Cargue convertida correctamente


## Listamos el schema del dataframe, y despues listamos las primeras 5 filas.

In [ ]:
df_residuos_sp.printSchema()

df_residuos_sp.show(5)

root
 |-- ID Empresa: string (nullable = true)
 |-- Empresa: string (nullable = true)
 |-- Año: string (nullable = true)
 |-- Mes: string (nullable = true)
 |-- NUSD: string (nullable = true)
 |-- Nombre del Sitio de Disposición Final: string (nullable = true)
 |-- Origen: string (nullable = true)
 |-- NUAP: string (nullable = true)
 |-- Nombre del área de Prestación: string (nullable = true)
 |-- Departamento: string (nullable = true)
 |-- Municipio: string (nullable = true)
 |-- Toneladas Dispuestas: double (nullable = true)
 |-- Fecha de Cargue: date (nullable = true)

+----------+--------------------+----+---+-----+-------------------------------------+---------+-----+-----------------------------+------------+-----------+--------------------+---------------+
|ID Empresa|             Empresa| Año|Mes| NUSD|Nombre del Sitio de Disposición Final|   Origen| NUAP|Nombre del área de Prestación|Departamento|  Municipio|Toneladas Dispuestas|Fecha de Cargue|
+----------+-------------------

In [ ]:
# ==========================================
# RANGO TEMPORAL
# ==========================================

#CONTEO POR AÑO
df_residuos_sp.groupBy("Año").count().orderBy("Año").show(23)

+----+-----+
| Año|count|
+----+-----+
|2006| 2866|
|2007| 8741|
|2008|10105|
|2009|10489|
|2010|10957|
|2011|11051|
|2012|11834|
|2013|12143|
|2014|12045|
|2015|12397|
|2016| 9909|
|2017| 7239|
|2018| 3280|
|2019| 3596|
|2020| 3272|
|2021| 2330|
|2022| 1549|
|2023| 1269|
|2024| 1071|
|2025|  542|
|2026|   31|
| NaN|    2|
+----+-----+



In [ ]:
# ==========================================
# VALIDACIÓN DE MESES
# ==========================================

df_residuos_sp.groupBy("Mes").count().orderBy("Mes").show(13)

+---+-----+
|Mes|count|
+---+-----+
|  1|11330|
| 10|11585|
| 11|11515|
| 12|11310|
|  2|11292|
|  3|11310|
|  4|11334|
|  5|11265|
|  6|11243|
|  7|11310|
|  8|11644|
|  9|11578|
|NaN|    2|
+---+-----+



In [ ]:
#eliminar NaN en año que al parecer son string "NaN"
df_residuos_sp = df_residuos_sp.filter(df_residuos_sp["Año"] != "NaN")

In [ ]:
df_residuos_sp.groupBy("Mes").count().orderBy("Mes").show(13)

+---+-----+
|Mes|count|
+---+-----+
|  1|11330|
| 10|11585|
| 11|11515|
| 12|11310|
|  2|11292|
|  3|11310|
|  4|11334|
|  5|11265|
|  6|11243|
|  7|11310|
|  8|11644|
|  9|11578|
+---+-----+



In [ ]:
# ==========================================
# FECHA ANALÍTICA DEL MODELO
# ==========================================

from pyspark.sql.functions import concat_ws, lit, to_date

df_residuos_sp = df_residuos_sp.withColumn(
    "fecha_modelo",

    to_date(
        concat_ws(
            "-",
            col("Año"),
            lpad(col("Mes"), 2, "0"),
            lit("01")
        ),
        "yyyy-MM-dd"
    )
)

print("Fecha analítica creada correctamente")

Fecha analítica creada correctamente


In [ ]:
# ==========================================
# VALIDACIÓN FECHA MODELO
# ==========================================

df_residuos_sp.select(
    "Año",
    "Mes",
    "fecha_modelo"
).show(14, truncate=False)

+----+---+------------+
|Año |Mes|fecha_modelo|
+----+---+------------+
|2006|8  |2006-08-01  |
|2006|9  |2006-09-01  |
|2006|10 |2006-10-01  |
|2006|11 |2006-11-01  |
|2006|12 |2006-12-01  |
|2007|1  |2007-01-01  |
|2007|2  |2007-02-01  |
|2007|3  |2007-03-01  |
|2007|4  |2007-04-01  |
|2007|5  |2007-05-01  |
|2007|6  |2007-06-01  |
|2007|7  |2007-07-01  |
|2007|8  |2007-08-01  |
|2007|9  |2007-09-01  |
+----+---+------------+
only showing top 14 rows


In [ ]:
df_modelo_residuos = df_residuos_sp.select(
    "ID Empresa",
    "Empresa",
    "Año",
    "Mes",
    "NUSD",
    "Nombre del Sitio de Disposición Final",
    "Origen",
    "NUAP",
    "Nombre del área de Prestación",
    "Departamento",
    "Municipio",
    "Toneladas Dispuestas",
    "Fecha de Cargue",
    "fecha_modelo"
)

print("Dataframe analítico creado correctamente")

# Vista inicial
df_modelo_residuos.show(5, truncate=False)

Dataframe analítico creado correctamente
+----------+------------------------------+----+---+-----+-------------------------------------+---------+-----+-----------------------------+------------+-----------+--------------------+---------------+------------+
|ID Empresa|Empresa                       |Año |Mes|NUSD |Nombre del Sitio de Disposición Final|Origen   |NUAP |Nombre del área de Prestación|Departamento|Municipio  |Toneladas Dispuestas|Fecha de Cargue|fecha_modelo|
+----------+------------------------------+----+---+-----+-------------------------------------+---------+-----+-----------------------------+------------+-----------+--------------------+---------------+------------+
|5         |MUNICIPIO DE CURILLO - CAQUETA|2006|8  |10493|BASURERO MUNICIPAL                   |NO APLICA|4714 |EL DONCELLO                  |CAQUETÁ     |EL DONCELLO|45.0                |2009-07-10     |2006-08-01  |
|5         |MUNICIPIO DE CURILLO - CAQUETA|2006|9  |10493|BASURERO MUNICIPAL           

In [ ]:
# ==========================================
# LIMPIEZA Y NORMALIZACIÓN Municipio
# ==========================================

# Lista de columnas que queremos normalizar
columnas_territoriales = ["Municipio", "Departamento"]

for columna in columnas_territoriales:
    df_modelo_residuos = df_modelo_residuos.withColumn(
        columna,
        # 1. Convertir a mayúsculas y quitar espacios a los extremos
        trim(upper(col(columna)))
    ).withColumn(
        columna,
        # 2. Quitar tildes y diéresis (como ya está en mayúsculas, solo buscamos esas)
        translate(col(columna), 'ÁÉÍÓÚÄËÏÖÜ', 'AEIOUAEIOU')
    ).withColumn(
        columna,
        # 3. Eliminar todo lo que NO (^) sea la A-Z, la Ñ o un espacio (\s)
        # Esto quitará puntos, comas, guiones, números y caracteres raros.
        regexp_replace(col(columna), r'[^A-ZÑ\s]', '')
    ).withColumn(
        columna,
        # 4. Reemplazar múltiples espacios generados por un solo espacio
        regexp_replace(col(columna), r'\s+', ' ')
    ).withColumn(
        columna,
        # 5. Un trim final por si al eliminar caracteres quedaron espacios en los bordes
        trim(col(columna))
    )

print("Normalización territorial completada")

Normalización territorial completada


In [ ]:
df_modelo_residuos.show(5, truncate=True)

+----------+--------------------+----+---+-----+-------------------------------------+---------+-----+-----------------------------+------------+-----------+--------------------+---------------+------------+
|ID Empresa|             Empresa| Año|Mes| NUSD|Nombre del Sitio de Disposición Final|   Origen| NUAP|Nombre del área de Prestación|Departamento|  Municipio|Toneladas Dispuestas|Fecha de Cargue|fecha_modelo|
+----------+--------------------+----+---+-----+-------------------------------------+---------+-----+-----------------------------+------------+-----------+--------------------+---------------+------------+
|         5|MUNICIPIO DE CURI...|2006|  8|10493|                   BASURERO MUNICIPAL|NO APLICA| 4714|                  EL DONCELLO|     CAQUETA|EL DONCELLO|                45.0|     2009-07-10|  2006-08-01|
|         5|MUNICIPIO DE CURI...|2006|  9|10493|                   BASURERO MUNICIPAL|NO APLICA|10064|         MUNICIPIO DE CURILLO|     CAQUETA|    CURILLO|           

In [ ]:
# ==========================================
# VALIDACIÓN DE NULOS
# ==========================================

df_modelo_residuos.select([

    count(
        when(col(c).isNull(), c)
    ).alias(c)

    for c in df_modelo_residuos.columns

]).show()

+----------+-------+---+---+----+-------------------------------------+------+----+-----------------------------+------------+---------+--------------------+---------------+------------+
|ID Empresa|Empresa|Año|Mes|NUSD|Nombre del Sitio de Disposición Final|Origen|NUAP|Nombre del área de Prestación|Departamento|Municipio|Toneladas Dispuestas|Fecha de Cargue|fecha_modelo|
+----------+-------+---+---+----+-------------------------------------+------+----+-----------------------------+------------+---------+--------------------+---------------+------------+
|         0|      0|  0|  0|   0|                                    0|     0|   0|                            0|           0|        0|                   0|              0|           0|
+----------+-------+---+---+----+-------------------------------------+------+----+-----------------------------+------------+---------+--------------------+---------------+------------+



In [ ]:
# ==========================================
# VALIDACIÓN DE MUNICIPIOS Y DEPARTAMENTOS
# ==========================================

print("Cantidad de departamentos únicos:")
print(
    df_modelo_residuos.select("Departamento")
    .distinct()
    .count()
)

print("Cantidad de municipios únicos:")
print(
    df_modelo_residuos.select("Municipio")
    .distinct()
    .count()
)

Cantidad de departamentos únicos:
34
Cantidad de municipios únicos:
896


In [ ]:
# ==========================================
# TOP MUNICIPIOS CON MÁS REGISTROS
# ==========================================

df_modelo_residuos.groupBy("Municipio") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)

+------------+-----+
|Municipio   |count|
+------------+-----+
|ND          |80935|
|BOGOTA DC   |415  |
|BUENAVISTA  |297  |
|MEDELLIN    |273  |
|CALI        |266  |
|CALARCA     |247  |
|BARRANQUILLA|222  |
|ARMENIA     |219  |
|NEIVA       |219  |
|NARINO      |207  |
|BUCARAMANGA |205  |
|RICAURTE    |204  |
|COROZAL     |202  |
|YUMBO       |202  |
|TAURAMENA   |198  |
|SANTACRUZ   |198  |
|SAN PEDRO   |197  |
|YOPAL       |188  |
|CUCUTA      |188  |
|ALBANIA     |187  |
+------------+-----+
only showing top 20 rows


In [ ]:
# ==========================================
# VALIDACIÓN DE TONELADAS
# ==========================================

df_modelo_residuos.select(

    min("Toneladas Dispuestas").alias("minimo"),
    max("Toneladas Dispuestas").alias("maximo"),
    avg("Toneladas Dispuestas").alias("promedio")

).show()

+------+----------+----------------+
|minimo|    maximo|        promedio|
+------+----------+----------------+
|   0.0|1.579114E7|7909.67733854565|
+------+----------+----------------+



In [ ]:
# ==========================================
# DETECCIÓN DE TONELADAS NEGATIVAS
# ==========================================

df_modelo_residuos.filter(
    col("Toneladas Dispuestas") < 0
).show()

+-------+-------------------------------------+----------+---+---+----+------+----+-----------------------------+------------+---------+--------------------+---------------+------------+
|Empresa|Nombre del Sitio de Disposición Final|ID Empresa|Año|Mes|NUSD|Origen|NUAP|Nombre del área de Prestación|Departamento|Municipio|Toneladas Dispuestas|Fecha de Cargue|fecha_modelo|
+-------+-------------------------------------+----------+---+---+----+------+----+-----------------------------+------------+---------+--------------------+---------------+------------+
+-------+-------------------------------------+----------+---+---+----+------+----+-----------------------------+------------+---------+--------------------+---------------+------------+



In [ ]:
# ==========================================
# TOP MUNICIPIOS CON MÁS TONELADAS
# ==========================================

df_modelo_residuos.groupBy("Municipio") \
    .agg(
        sum("Toneladas Dispuestas").alias("total_toneladas")
    ) \
    .orderBy(desc("total_toneladas")) \
    .show(20, truncate=False)

+-------------------+--------------------+
|Municipio          |total_toneladas     |
+-------------------+--------------------+
|BUCARAMANGA        |8.013265882179997E8 |
|BOSCONIA           |7.713663265E7       |
|BARRANCABERMEJA    |7.628437966000001E7 |
|BOGOTA DC          |2.272982216699999E7 |
|                   |1.2686629510000044E7|
|YARUMAL            |9353206.171199994   |
|CALI               |6071276.310000003   |
|ZIPAQUIRA          |5313041.639999989   |
|BARBOSA            |5185082.475800004   |
|BARRANQUILLA       |4610008.269000007   |
|POPAYAN            |4217708.6999999955  |
|CARTAGENA DE INDIAS|4204014.129800002   |
|CHIQUINQUIRA       |3148693.0372        |
|GIRON              |2407472.3819999993  |
|CUCUTA             |2325411.439999998   |
|GUADALAJARA DE BUGA|1923595.0389999964  |
|SANTA ROSA DE CABAL|1644612.4530000011  |
|MANIZALES          |1619075.7226999998  |
|RETIRO             |1616605.9099999985  |
|MEDELLIN           |1513496.3800000001  |
+----------

In [ ]:
# Nota: Si ya ejecutaste el código anterior, los "ND" ahora son valores nulos.
# Usaremos una condición que atrape ambos casos por seguridad.
condicion_nd = col("Municipio").isNull() | (col("Municipio") == "ND")
df_nd = df_modelo_residuos.filter(condicion_nd)

# ==========================================
# 1. VISTAZO GENERAL
# ==========================================
print("Muestra de registros sin Municipio:")
df_nd.select("Año", "Departamento", "Municipio", "Empresa", "Nombre del Sitio de Disposición Final", "Toneladas Dispuestas").show(10, truncate=False)

# ==========================================
# 2. ¿QUIÉN ESTÁ REPORTANDO ESTOS DATOS VACÍOS?
# ==========================================
print("Agrupación por Empresa y Sitio de Disposición:")
df_nd.groupBy("Departamento", "Empresa", "Nombre del Sitio de Disposición Final") \
     .count() \
     .orderBy(col("count").desc()) \
     .show(20, truncate=False)

# ==========================================
# 3. ¿CUÁNDO OCURRIÓ ESTO?
# ==========================================
print("Distribución de ND por Año:")
df_nd.groupBy("Año").count().orderBy("Año").show()

Muestra de registros sin Municipio:
+---+------------+---------+-------+-------------------------------------+--------------------+
|Año|Departamento|Municipio|Empresa|Nombre del Sitio de Disposición Final|Toneladas Dispuestas|
+---+------------+---------+-------+-------------------------------------+--------------------+
+---+------------+---------+-------+-------------------------------------+--------------------+

Agrupación por Empresa y Sitio de Disposición:
+------------+-------+-------------------------------------+-----+
|Departamento|Empresa|Nombre del Sitio de Disposición Final|count|
+------------+-------+-------------------------------------+-----+
+------------+-------+-------------------------------------+-----+

Distribución de ND por Año:
+---+-----+
|Año|count|
+---+-----+
+---+-----+



In [ ]:
from pyspark.sql.functions import col, coalesce, first

# ==========================================
# 1. CREAR EL DICCIONARIO DE REFERENCIA
# ==========================================
# Filtramos solo los que están bien llenos para usarlos como fuente de verdad
df_referencia = df_modelo_residuos.filter(
    (col("Municipio") != "ND") &
    (col("Departamento") != "ND") &
    (col("Municipio").isNotNull())
).select(
    "Empresa",
    "Nombre del Sitio de Disposición Final",
    "Departamento",
    "Municipio"
).distinct()

# Nota: Si una empresa/sitio tiene varios municipios (raro),
# nos quedamos con el primero para evitar duplicar filas en el join
df_referencia = df_referencia.groupBy("Empresa", "Nombre del Sitio de Disposición Final") \
    .agg(
        first("Departamento").alias("Dep_Ref"),
        first("Municipio").alias("Mun_Ref")
    )

# ==========================================
# 2. UNIR Y REPARAR LOS "ND"
# ==========================================

# Hacemos un Left Join de nuestro dataframe original con la referencia
df_modelo_residuos = df_modelo_residuos.join(
    df_referencia,
    on=["Empresa", "Nombre del Sitio de Disposición Final"],
    how="left"
)

# Usamos coalesce: "Si el original es ND o nulo, usa el de referencia"
df_modelo_residuos = df_modelo_residuos.withColumn(
    "Municipio",
    when((col("Municipio") == "ND") | (col("Municipio").isNull()), col("Mun_Ref"))
    .otherwise(col("Municipio"))
).withColumn(
    "Departamento",
    when((col("Departamento") == "ND") | (col("Departamento").isNull()), col("Dep_Ref"))
    .otherwise(col("Departamento"))
)

# Limpiamos las columnas temporales del join
df_modelo_residuos = df_modelo_residuos.drop("Dep_Ref", "Mun_Ref")

print("¡Rescate de datos completado!")

¡Rescate de datos completado!


In [ ]:
df_modelo_residuos.groupBy("Municipio") \
    .agg(
        sum("Toneladas Dispuestas").alias("total_toneladas")
    ) \
    .orderBy(desc("total_toneladas")) \
    .show(20, truncate=False)

+-------------------+--------------------+
|Municipio          |total_toneladas     |
+-------------------+--------------------+
|BUCARAMANGA        |8.013265882179997E8 |
|BOSCONIA           |7.713663265E7       |
|BARRANCABERMEJA    |7.628437966000001E7 |
|BOGOTA DC          |2.272982216699999E7 |
|NULL               |1.2688966310000045E7|
|YARUMAL            |9353206.171199994   |
|CALI               |6071276.310000003   |
|ZIPAQUIRA          |5313041.639999989   |
|BARBOSA            |5185082.475800004   |
|BARRANQUILLA       |4610008.269000007   |
|POPAYAN            |4217708.6999999955  |
|CARTAGENA DE INDIAS|4204014.129800002   |
|CHIQUINQUIRA       |3148693.0372        |
|GIRON              |2407472.3819999993  |
|CUCUTA             |2325411.439999998   |
|GUADALAJARA DE BUGA|1923595.0389999964  |
|SANTA ROSA DE CABAL|1644612.4530000011  |
|MANIZALES          |1619075.7226999998  |
|RETIRO             |1616605.9099999985  |
|MEDELLIN           |1513496.3800000001  |
+----------

In [ ]:
# Comprobamos cuántos municipios siguen siendo nulos o ND después del JOIN
final_nd = df_modelo_residuos.filter(
    (col("Municipio").isNull()) | (col("Municipio") == "ND")
).count()

print(f"Registros originales ND: 80935")
print(f"Registros que NO se pudieron rescatar: {final_nd}")
print(f"Porcentaje de éxito: {((80935 - final_nd) / 80935) * 100:.2f}%")

Registros originales ND: 80935
Registros que NO se pudieron rescatar: 12464
Porcentaje de éxito: 84.60%


In [ ]:
# Filtramos lo que aún es ND
df_huerfanos = df_modelo_residuos.filter(
    (col("Municipio").isNull()) | (col("Municipio") == "ND")
)

print("Top 20 de Empresas/Sitios que siguen como ND:")
df_huerfanos.groupBy("Empresa", "Nombre del Sitio de Disposición Final") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(20, truncate=False)

Top 20 de Empresas/Sitios que siguen como ND:
+--------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------+-----+
|Empresa                                                                                                                                           |Nombre del Sitio de Disposición Final    |count|
+--------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------+-----+
|AGUAS DE PUERTO CAICEDO S.A.S E.S.P                                                                                                               |LA ESMERALDA                             |283  |
|EMPRESA PRESTADORA DEL SERVICIO PUBLICO DE ASEO DE SAN JACINTO S.A. E.S.P.                                                                        |RELLENO REGIONAL S

In [ ]:
from pyspark.sql.functions import regexp_extract, split

# Intentamos extraer Municipio y Departamento del nombre de la Empresa
# Buscamos el patrón: "MUNICIPIO DE [NOMBRE] - [DEP]"
df_modelo_residuos = df_modelo_residuos.withColumn(
    "Municipio",
    when(
        (col("Municipio") == "ND") | (col("Municipio").isNull()) | (col("Municipio") == ""),
        regexp_extract(col("Empresa"), r"MUNICIPIO DE\s+([A-Z\s]+?)\s+-", 1)
    ).otherwise(col("Municipio"))
).withColumn(
    "Departamento",
    when(
        (col("Departamento") == "ND") | (col("Departamento").isNull()) | (col("Departamento") == ""),
        regexp_extract(col("Empresa"), r"-\s+([A-Z\s]+)$", 1)
    ).otherwise(col("Departamento"))
)

# Limpiamos espacios que hayan quedado en la extracción
df_modelo_residuos = df_modelo_residuos.withColumn("Municipio", trim(col("Municipio"))) \
                                       .withColumn("Departamento", trim(col("Departamento")))

In [ ]:
# Comprobamos cuántos municipios siguen siendo nulos, ND o cadenas vacías después del JOIN
final_nd2 = df_modelo_residuos.filter(
    (col("Municipio").isNull()) | (col("Municipio") == "ND") | (col("Municipio") == "")
).count()

print(f"Registros originales ND (y/o vacíos): 12464 (estimado)")
print(f"Registros de Municipio que aún no se pudieron rescatar (nulos, 'ND' o vacíos): {final_nd2}")

# Calculamos el porcentaje de éxito basándonos en la reducción de registros problemáticos
# Nota: '12464' es el número de registros que eran ND o nulos antes de la segunda fase de rescate.
# Ahora que incluimos vacíos, el número base de 'problemas' podría ser mayor si no se transformaron a ND/nulos antes.
# Para una métrica más precisa, necesitaríamos un conteo inicial de ND/nulos/vacíos. Usaremos 12464 como base para esta iteración.
if 12464 > 0: # Evitar división por cero
    porcentaje_exito = ((12464 - final_nd2) / 12464) * 100
    print(f"Porcentaje de éxito (reducción de nulos/'ND'/vacíos): {porcentaje_exito:.2f}%")
else:
    print("No hubo registros originales 'ND' o nulos que intentar rescatar en la segunda fase.")

In [ ]:
# Comprobamos cuántos municipios siguen siendo nulos, ND o cadenas vacías después del JOIN
final_nd2 = df_modelo_residuos.filter(
    (col("Municipio").isNull()) | (col("Municipio") == "ND") | (col("Municipio") == "")
).count()

print(f"Registros originales ND (y/o vacíos): 12464 (estimado)")
print(f"Registros de Municipio que aún no se pudieron rescatar (nulos, 'ND' o vacíos): {final_nd2}")

# Calculamos el porcentaje de éxito basándonos en la reducción de registros problemáticos
# Nota: '12464' es el número de registros que eran ND o nulos antes de la segunda fase de rescate.
# Ahora que incluimos vacíos, el número base de 'problemas' podría ser mayor si no se transformaron a ND/nulos antes.
# Para una métrica más precisa, necesitaríamos un conteo inicial de ND/nulos/vacíos. Usaremos 12464 como base para esta iteración.
if 12464 > 0: # Evitar división por cero
    porcentaje_exito = ((12464 - final_nd2) / 12464) * 100
    print(f"Porcentaje de éxito (reducción de nulos/'ND'/vacíos): {porcentaje_exito:.2f}%")
else:
    print("No hubo registros originales 'ND' o nulos que intentar rescatar en la segunda fase.")

Registros originales ND: 12464
Registros que NO se pudieron rescatar: 0
Porcentaje de éxito: 100.00%


In [ ]:
#conocer el tamaño completo del df
df_modelo_residuos.count()


136716

In [ ]:
ruta_residuos_limpios = "/content/drive/MyDrive/26ET3/PROYECTOII/data_residuos_limpios.csv"

In [ ]:
#df_modelo_residuos.write.csv(
#    ruta_residuos_limpios,
#    header=True,
#    mode="overwrite"
#)

#print(f"DataFrame guardado exitosamente en: {ruta_residuos_limpios}")

DataFrame guardado exitosamente en: /content/drive/MyDrive/26ET3/PROYECTOII/data_residuos_limpios.csv


In [ ]:
df_modelo_residuos.show(5, truncate=False)

+----------+------------------------------+----+---+-----+-------------------------------------+---------+-----+-----------------------------+------------+-----------+--------------------+---------------+------------+
|ID Empresa|Empresa                       |Año |Mes|NUSD |Nombre del Sitio de Disposición Final|Origen   |NUAP |Nombre del área de Prestación|Departamento|Municipio  |Toneladas Dispuestas|Fecha de Cargue|fecha_modelo|
+----------+------------------------------+----+---+-----+-------------------------------------+---------+-----+-----------------------------+------------+-----------+--------------------+---------------+------------+
|5         |MUNICIPIO DE CURILLO - CAQUETA|2006|8  |10493|BASURERO MUNICIPAL                   |NO APLICA|4714 |EL DONCELLO                  |CAQUETÁ     |EL DONCELLO|45.0                |2009-07-10     |2006-08-01  |
|5         |MUNICIPIO DE CURILLO - CAQUETA|2006|9  |10493|BASURERO MUNICIPAL                   |NO APLICA|10064|MUNICIPIO DE CUR

In [ ]:
# Convertir el DataFrame de PySpark a Pandas DataFrame
df_pandas = df_modelo_residuos.toPandas()

# Guardar el DataFrame de Pandas como un único archivo CSV
df_pandas.to_csv(ruta_residuos_limpios, index=False, encoding='utf-8')

print(f"DataFrame guardado exitosamente como un único archivo CSV en: {ruta_residuos_limpios}")

DataFrame guardado exitosamente como un único archivo CSV en: /content/drive/MyDrive/26ET3/PROYECTOII/data_residuos_limpios.csv
